In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import copy

In [3]:
df = pd.read_csv('diabetes_prediction_dataset.csv')
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [6]:
le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'])
df['smoking_history'] = le.fit_transform(df['smoking_history'])

X = df.iloc[:,:-1]
y = df['diabetes']

X.size

800000

In [8]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
num_clients = 5
client_data_split = np.array_split(np.arange(len(X_train)), num_clients)

clients_X = [X_train[idx] for idx in client_data_split]
clients_y = [y_train.iloc[idx].values for idx in client_data_split]

In [10]:
global_model = SGDClassifier(loss='log_loss', max_iter=1, warm_start=True, random_state=42)
global_model.fit(X_train[:10], y_train[:10])

num_rounds = 5

c:\Users\mayur\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:726: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


In [11]:
for round_idx in range(num_rounds):
    local_weights = []
    local_intercepts = []
    
    for client_idx in range(num_clients):
        local_model = copy.deepcopy(global_model)
        local_model.partial_fit(clients_X[client_idx], clients_y[client_idx], classes=np.unique(y))
        local_weights.append(local_model.coef_)
        local_intercepts.append(local_model.intercept_)
    
    avg_weights = np.mean(local_weights, axis=0)
    avg_intercept = np.mean(local_intercepts, axis=0)
    
    global_model.coef_ = avg_weights
    global_model.intercept_ = avg_intercept
    
    y_pred = global_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Round {round_idx + 1}: Global Model Accuracy = {accuracy:.4f}")

Round 1: Global Model Accuracy = 0.9478
Round 2: Global Model Accuracy = 0.9484
Round 3: Global Model Accuracy = 0.9478
Round 4: Global Model Accuracy = 0.9478
Round 5: Global Model Accuracy = 0.9479
